[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanenalmayouf/applied-ml-workshop/blob/main/labs_colab/day5/final_project_manafeth_churn_solution.ipynb)

# 🏁 مختبر اليوم الخامس — مشروع منافذ المتكامل وفحص تغير التشغيل

**ورشة أسس تعلم الآلة التطبيقي — اليوم 5 من 5**

هذا الدفتر مبني على مواصفات مختبرات «منافذ» المعتمدة للدورة، ومُجهَّز للعمل مباشرة في **Google Colab** أو في Jupyter محليًا.

**كيف تفتحه في Colab:** من قائمة `File → Upload notebook` في Colab ارفع هذا الملف. إذا لم يجد الدفتر مجلد البيانات تلقائيًا، ستظهر لك خانة لرفع ملف حزمة البيانات (`manafeth_data_package.zip`) المرفق بجوار هذا الدفتر — ارفعه وسيُستكمل التحميل تلقائيًا.

> 📌 راجع `00_start_here.md` قبل البدء لمعرفة طريقة استخدام خلايا **فكّر أولًا** و**TODO** و**مساعدة** في هذه الدفاتر.

In [ ]:
from pathlib import Path
import pandas as pd

# 1) نبحث عن مجلد البيانات بجانب هذا الدفتر (يعمل محليًا وفي Colab إذا رفعت المجلد كاملًا)
DATA_CANDIDATES = [Path("manafeth_data_package"), Path("data"), Path("../data/raw"), Path("data/raw")]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), None)

# 2) إذا لم نجد المجلد ونحن داخل Google Colab، نطلب من الطالب رفع حزمة البيانات (ملف zip)
if DATA_DIR is None:
    try:
        from google.colab import files
        import zipfile

        print("لم يتم العثور على مجلد البيانات محليًا.")
        print("ارفع ملف حزمة البيانات (.zip) الذي يرافق هذا المختبر ثم انتظر انتهاء الرفع...")
        uploaded = files.upload()
        for name in uploaded:
            if name.lower().endswith(".zip"):
                with zipfile.ZipFile(name) as z:
                    z.extractall(".")
        DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), Path("manafeth_data_package"))
    except ImportError:
        DATA_DIR = Path("manafeth_data_package")
        print("تنبيه: لسنا داخل Google Colab ولم يوجد مجلد بيانات — ضع حزمة البيانات بجانب الدفتر.")

CUSTOMERS_PATH = DATA_DIR / "manafeth_customers.parquet"
ORDERS_PATH = DATA_DIR / "manafeth_orders.parquet"
VEHICLES_PATH = DATA_DIR / "markabat_listings_sample.csv"
SHIFTED_PATH = DATA_DIR / "shifted_month.parquet"

print("مجلد البيانات المستخدم:", DATA_DIR.resolve())
assert CUSTOMERS_PATH.exists(), "تعذّر العثور على manafeth_customers.parquet — تأكد من رفع حزمة البيانات كاملة."

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (7, 4)

## 🎯 هدف المختبر

تنجز اليوم مشروع تعلم آلة متكاملًا من البداية إلى النهاية على بيانات منافذ الفعلية، ثم تجري فحصًا تشغيليًا أوليًا على شهر لاحق. يجمع المشروع: صياغة المشكلة، تجهيز البيانات، المقارنة، التقييم، تحليل الأخطاء، والتواصل المسؤول عن النتائج.

## السيناريو والبيانات

تبني نموذجًا يرتب العملاء الذين يُحتمل أن يغادروا خلال 30 يومًا من تاريخ لقطة شهرية، باستخدام `manafeth_customers.parquet` للتطوير والتقييم التاريخي. بعد تثبيت النموذج، تستخدم `shifted_month.parquet` — جدول من **شهر لاحق بلا هدف معروف** — للمقارنة فقط، لا للتقييم.

> ⚠️ **تنبيه مهم:** لا تُدخل أيًا من الأعمدة `refund_issued` أو `support_ticket_after_snapshot` أو `next_month_orders` في المشروع. ملف الشهر اللاحق لا يحتوي على الهدف إطلاقًا؛ لذلك **لا يجوز** كتابة دقة أو استدعاء له — فقط نسبة تجاوز العتبة.

## 🤔 فكّر أولًا

إذا ارتفعت نسبة العملاء الذين يتجاوزون عتبة الاتصال في الشهر اللاحق مقارنة بشهر التطوير، هل هذا بالضرورة يعني أن النموذج «تحسّن» أو «ساء»؟ ما الذي يمكن أن يكون قد تغيّر غير جودة النموذج نفسه؟

### المرحلة 1 — بطاقة المشروع والاستكشاف

**بطاقة المشروع (أكمل بصياغتك):**

- **السؤال:** هل سيغادر العميل خلال 30 يومًا من تاريخ اللقطة؟
- **وحدة التنبؤ:** ………
- **الهدف:** ………
- **قرار المتابعة:** ………
- **الخصائص المتاحة عند تاريخ اللقطة:** ………

In [ ]:
customers = pd.read_parquet(CUSTOMERS_PATH)
customers.info()
customers.isna().sum().sort_values(ascending=False)

جدول القرار (آمن / معرّف / تسريب) — أعد استخدام تصنيفك من اليوم الأول:

In [ ]:
safe_features = [
    "city", "city_tier", "device", "payment_method",
    "tenure_months", "orders_per_month", "avg_basket_sar",
    "days_since_last_order", "distinct_categories", "promo_usage_rate",
    "avg_rating", "last_promo_used"
]
leak_columns = ["refund_issued", "support_ticket_after_snapshot", "next_month_orders"]
print("خصائص آمنة:", len(safe_features), "| أعمدة تسريب مستبعدة:", leak_columns)

### 📊 رسم: توزيع الهدف

مخطط أعمدة بعنوان ومحاور عربية — مثال مُنفَّذ من اليوم الأول.

In [ ]:
counts = customers["churned_30d"].value_counts().sort_index()
plt.bar(["استمر (0)", "غادر (1)"], counts.values, color=["#33CC99", "#F26522"])
plt.title("توزيع هدف مغادرة العملاء خلال 30 يومًا")
plt.ylabel("عدد العملاء")
plt.show()

### المرحلة 2 — التجهيز والمقارنة

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, precision_recall_curve, ConfusionMatrixDisplay
from xgboost import XGBClassifier

numeric_features = [
    "city_tier", "tenure_months", "orders_per_month", "avg_basket_sar",
    "days_since_last_order", "distinct_categories", "promo_usage_rate",
    "avg_rating"
]
categorical_features = ["city", "device", "payment_method", "last_promo_used"]

X = customers[safe_features]
y = customers["churned_30d"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor = ColumnTransformer([
    ("numbers", Pipeline([
        ("fill", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler())
    ]), numeric_features),
    ("categories", Pipeline([
        ("fill", SimpleImputer(strategy="constant", fill_value="غير_معروف")),
        ("encode", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

In [ ]:
models = {
    "انحدار لوجستي": LogisticRegression(max_iter=1000),
    "غابة عشوائية": RandomForestClassifier(n_estimators=150, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=3, eval_metric="logloss", random_state=42)
}
comparison = []
for name, model in models.items():
    workflow = Pipeline([("prepare", preprocessor), ("model", model)])
    scores = cross_val_score(workflow, X_train, y_train, cv=5, scoring="average_precision")
    comparison.append([name, scores.mean(), scores.std()])

results_df = pd.DataFrame(
    comparison, columns=["النموذج", "متوسط الدقة", "تغير النتيجة بين الطيات"]
).sort_values("متوسط الدقة", ascending=False)
results_df

**اكتب هنا سبب اختيارك للنموذج قبل أن تفتح بيانات الاختبار في الخلية التالية:**

_سبب الاختيار: ……_

### المرحلة 3 — التقييم والتفسير

In [ ]:
chosen_name = results_df.iloc[0]["النموذج"]
final_workflow = Pipeline([("prepare", preprocessor), ("model", models[chosen_name])])
final_workflow.fit(X_train, y_train)
probabilities = final_workflow.predict_proba(X_test)[:, 1]

ap = average_precision_score(y_test, probabilities)
precision, recall, _ = precision_recall_curve(y_test, probabilities)
plt.plot(recall, precision, color="#5B4FCF")
plt.xlabel("الاستدعاء"); plt.ylabel("الإحكام"); plt.title("منحنى الدقة–الاستدعاء")
plt.show()

contact_count = int(len(y_test) * 0.20)
top_customers = pd.DataFrame({
    "الحقيقة": y_test.to_numpy(), "احتمال_المغادرة": probabilities
}).sort_values("احتمال_المغادرة", ascending=False).head(contact_count)
recall_at_20 = top_customers["الحقيقة"].sum() / y_test.sum()
print("متوسط الدقة على الاختبار:", round(ap, 3))
print("الاستدعاء بين أعلى 20%:", round(recall_at_20, 3))

predictions_at_50 = (probabilities >= 0.50).astype(int)
ConfusionMatrixDisplay.from_predictions(
    y_test, predictions_at_50, display_labels=["استمر (0)", "غادر (1)"], cmap="Purples"
)
plt.title("مصفوفة الالتباس عند عتبة 0.50")
plt.show()

In [ ]:
review = X_test.copy()
review["الحقيقة"] = y_test.to_numpy()
review["احتمال_المغادرة"] = probabilities
review["التوقع_عند_0_50"] = predictions_at_50
errors = review[review["الحقيقة"] != review["التوقع_عند_0_50"]]
errors.head()

**اكتب ملاحظة واحدة تستند إلى البيانات فقط عن نمط تكرر في صفوف الخطأ:**

_الملاحظة: ……_

### المرحلة 4 — فحص تغير التشغيل في شهر لاحق

In [ ]:
import numpy as np

# عتبة تُختار من بيانات الاختبار لتحديد أعلى 20% من العملاء بالترتيب
contact_threshold = np.quantile(probabilities, 0.80)

shifted = pd.read_parquet(SHIFTED_PATH)
X_shifted = shifted[safe_features]
shifted_probabilities = final_workflow.predict_proba(X_shifted)[:, 1]
shifted_contact_rate = (shifted_probabilities >= contact_threshold).mean()

print("عتبة الاتصال:", round(contact_threshold, 3))
print("نسبة عملاء شهر التطوير الأصلية (تعريفًا):", 0.20)
print("نسبة العملاء المتجاوزين للعتبة في الشهر اللاحق:", round(shifted_contact_rate, 3))

### 📊 رسم: مقارنة نسبة التواصل — شهر التطوير مقابل الشهر اللاحق

مثال مُنفَّذ — مخطط عمودين بسيط للمقارنة البصرية بين النسبة المرجعية (20% تعريفًا) ونسبة الشهر اللاحق الفعلية.

In [ ]:
plt.bar(["شهر التطوير (مرجعي)", "الشهر اللاحق (فعلي)"],
        [0.20, shifted_contact_rate], color=["#8C8C8C", "#F26522"])
plt.title("نسبة العملاء الذين يتجاوزون عتبة الاتصال")
plt.ylabel("النسبة")
plt.show()

**اكتب تنبيهًا تشغيليًا مناسبًا (مثال البداية):**

"تغيرت نسبة التنبيهات من 20% إلى ___%، لذا يجب مراجعة القدرة التشغيلية لفريق خدمة العملاء وقياس النتائج الفعلية عند توفرها. هذا **ليس** حكمًا على دقة النموذج لأن الشهر اللاحق لا يحتوي على هدف معروف."

## النتيجة المتوقعة

دفتر مشروع منظم يضم: تعريف المشكلة، فحص البيانات، خط معالجة، مقارنة نماذج، جدول نتائج، رسمًا واحدًا على الأقل، منحنى دقة–استدعاء، مصفوفة التباس، تحليل أخطاء، وفحص تغير نسبة التنبيهات في الشهر اللاحق. **تغيّر النسبة ليس «دقة» ولا «فشلًا» للنموذج** — هو إشارة تشغيلية تحتاج قياس نتائج فعلية لاحقًا.

## المهارات التي راجعتها

سير عمل تعلم الآلة كاملًا: صياغة المشكلة، منع التسريب، pandas وJupyter، المعالجة المسبقة، التصنيف، التحقق المتقاطع، اختيار مقياس مناسب، Matplotlib، XGBoost، تحليل الأخطاء، وتقديم نتيجة مسؤولة.

## ✅ تحقق ذاتيًا قبل إغلاق الدفتر
- [ ] لم تدخل أي من أعمدة التسريب الثلاثة إلى `X`
- [ ] قارنت 3 نماذج على الأقل بالتحقق المتقاطع، واخترت قبل فتح بيانات الاختبار
- [ ] لم تكتب دقة أو استدعاء لملف `shifted_month.parquet` — فقط نسبة تجاوز عتبة
- [ ] لديك جملة تحليل أخطاء وجملة تنبيه تشغيلي مبنيتان على البيانات فقط

**مبروك — أكملت المشروع المتكامل لورشة أسس تعلم الآلة التطبيقي على بيانات منافذ الحقيقية.**